# 05 Post-Processing — Mindestfläche, 10 GWh, 500 kWh/kW

**PA1 ZHAW IUNR** | Bächler, Haag, Reichlin | Betreuer: Patrick Laube

Filtert die WLC-Eignungskarte nach den gesetzlichen Anforderungen (Art. 71a EnG):
1. Mindestfläche ≥ 10 ha zusammenhängend
2. Min. 10 GWh Jahresproduktion
3. Min. 500 kWh/kW Winterproduktion

**Input:**
```
outputs/suitability_wlc.tif              (WLC-Eignungskarte, kontinuierlich 0–1)
outputs/suitability_classes.tif          (5 Klassen)
data/processed/constraints/constraint_mask_s2.tif
```

**Output:**
```
outputs/suitability_postprocessed.tif    (gefiltert, 0 = entfernt)
outputs/top_sites.gpkg                   (Top-Standorte als Polygone)
outputs/tables/postprocessing_stats.csv
outputs/figures/postprocessing_map.png
```

## 1. Setup & Imports

In [ ]:
from pathlib import Path                                                    # Cross-platform file path handling
import geopandas as gpd                                                    # Geospatial vector data manipulation
import numpy as np                                                         # Numerical array operations and math
import pandas as pd                                                        # Tabular data handling and analysis
import rasterio                                                            # Read/write raster GIS data (GeoTIFF)
from rasterio.features import shapes as rio_shapes                         # Raster-to-vector polygon extraction
from scipy.ndimage import label                                            # Connected component labeling
from shapely.geometry import shape as shp_shape                            # Convert GeoJSON geometry to Shapely
import matplotlib.pyplot as plt                                            # Plotting and visualization library
import matplotlib.colors as mcolors                                        # Custom colormaps and color normalization
import warnings
warnings.filterwarnings("ignore")                                          # Suppress non-critical warnings

# Load data paths
RAW  = Path("../data/raw")                                                 # Raw input data directory
PROC = Path("../data/processed")                                           # Processed data directory
OUT  = Path("../outputs")                                                  # Output directory
for d in [OUT, OUT / "figures", OUT / "tables"]:
    d.mkdir(parents=True, exist_ok=True)                                   # Create output subdirectories if not existing

# Define constants
CRS    = "EPSG:2056"                                                       # Target CRS (LV95, Swiss standard)
NODATA = -9999.0                                                           # NoData value for raster pixels
RES    = 25                                                                # Raster resolution in meters
PX_AREA_M2  = RES * RES                                                   # Pixel area: 625 m²
PX_AREA_HA  = PX_AREA_M2 / 1e4                                            # Pixel area: 0.0625 ha
PX_AREA_KM2 = PX_AREA_M2 / 1e6                                            # Pixel area: 0.000625 km²

# Post-Processing thresholds (Art. 71a EnG, Swissolar 2025)
MIN_AREA_HA        = 10                                                    # Min. contiguous area [ha]
MIN_ANNUAL_GWH     = 10                                                    # Min. annual production [GWh]
MIN_WINTER_KWH_KW  = 500                                                  # Min. winter production [kWh/kWp]
ETA                = 0.18                                                  # System efficiency (conservative, alpine)

print("✓ Setup OK")
print(f"  Thresholds: ≥{MIN_AREA_HA} ha | ≥{MIN_ANNUAL_GWH} GWh/a | ≥{MIN_WINTER_KWH_KW} kWh/kW | η={ETA}")

## 2. Weights & Classes

In [ ]:
# Suitability classes (identical to 04_wlc — keep synchronized)
CLASSES = {
    1: (0.0, 0.6, "Unsuitable or Low suitability"),                        # Class 1: 0.0–0.6
    2: (0.6, 0.7, "Moderate suitability"),                                 # Class 2: 0.6–0.7
    3: (0.7, 0.8, "Suitable"),                                             # Class 3: 0.7–0.8
    4: (0.8, 0.9, "Well suited"),                                          # Class 4: 0.8–0.9
    5: (0.9, 1.0, "Very well suited"),                                     # Class 5: 0.9–1.0
}

# Colormap (colorblind-safe, identical to 04_wlc)
CLASS_COLORS = ["#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026"]     # Blue → Red diverging
CLASS_LABELS = [v[2] for v in CLASSES.values()]                            # Label strings for legend
cmap_cls     = mcolors.ListedColormap(CLASS_COLORS)                        # Discrete colormap
norm_cls     = mcolors.BoundaryNorm([0.5,1.5,2.5,3.5,4.5,5.5], cmap_cls.N)  # Class boundaries

def suit_to_class(arr, mask_2d):
    """Classify continuous suitability [0–1] into 5 discrete classes."""
    c = np.zeros(arr.shape, dtype=np.float32)                              # Init with 0 (no class)
    m = mask_2d & (arr > 0)                                                # Valid pixels only
    v = arr[m]                                                             # Extract valid values
    cls = np.ones(v.shape)                                                 # Default: class 1
    for cls_id, (lo, hi, _) in CLASSES.items():                            # Assign higher classes
        if cls_id > 1:
            cls[v >= lo] = cls_id                                          # Overwrite with higher class
    c[m] = cls                                                             # Write back to array
    return np.where(m, c, np.nan)                                          # NaN outside mask

def mean_suit_to_class(s):
    """Return class (1–5) for a single suitability value."""
    for cls_id in sorted(CLASSES.keys(), reverse=True):                    # Check from highest class
        lo, _, _ = CLASSES[cls_id]                                         # Lower bound
        if s >= lo:                                                        # First match wins
            return cls_id
    return min(CLASSES.keys())                                             # Fallback: lowest class

print(f"✓ Classes: { {k: v[2] for k, v in CLASSES.items()} }")

## 3. Data Loading

In [ ]:
print("=== Loading suitability map & constraint mask ===\n")

# WLC suitability map (continuous 0–1)
with rasterio.open(OUT / "suitability_wlc.tif") as src:                    # Open WLC output
    suit = src.read(1)                                                     # Read band 1
    transform = src.transform                                              # Affine transformation
    profile = src.profile.copy()                                           # GeoTIFF profile for outputs
    height, width = suit.shape                                             # Raster dimensions

# Constraint mask (binary 0/1)
with rasterio.open(PROC / "constraints/constraint_mask_s2.tif") as src:    # Open constraint mask
    mask = src.read(1)                                                     # Read band 1

valid = (suit > 0) & (mask == 1)                                           # Pixels with suitability inside mask
print(f"  WLC map:        {width}×{height}")
print(f"  Suitable pixels: {valid.sum():,} ({valid.sum() * PX_AREA_KM2:.0f} km²)")

In [ ]:
print("\n=== Loading solar radiation (absolute values) ===\n")

# Normalized criteria [0,1] → de-normalize to absolute kWh values
with rasterio.open(PROC / "criteria/f01_globalstrahlung.tif") as src:      # Annual global radiation
    f01_norm = src.read(1).astype(np.float32)                              # Normalized [0,1]

with rasterio.open(PROC / "criteria/f02_wintereinstrahlung.tif") as src:   # Winter radiation proxy
    f02_norm = src.read(1).astype(np.float32)                              # Normalized [0,1]

# De-normalization bounds (from 03_preprocessing_wlc — GR at 45° tilt)
# Formula: x_abs = x_norm * (max - min) + min
SOLAR_MIN,  SOLAR_MAX  = 169.0, 1787.0                                    # kWh/m²/a annual
WINTER_MIN, WINTER_MAX =  41.0,  981.0                                    # kWh/m² Oct–Mar

solar_abs  = f01_norm * (SOLAR_MAX  - SOLAR_MIN) + SOLAR_MIN              # Absolute annual [kWh/m²/a]
winter_abs = f02_norm * (WINTER_MAX - WINTER_MIN) + WINTER_MIN            # Absolute winter [kWh/m²]
solar_abs  = np.where(valid, solar_abs,  NODATA).astype(np.float32)       # Mask invalid pixels
winter_abs = np.where(valid, winter_abs, NODATA).astype(np.float32)       # Mask invalid pixels

print(f"  Solar annual:  {solar_abs[valid].min():.0f}–{solar_abs[valid].max():.0f} kWh/m²/a")
print(f"  Solar winter:  {winter_abs[valid].min():.0f}–{winter_abs[valid].max():.0f} kWh/m²")

## 4. Filter 1: Min. Area ≥ 10 ha

Connected-component labeling to identify contiguous areas, remove clusters below threshold.

Ref: Swissolar (2025) §4.1.1: Min. 7 MW → 10–20 ha net area for 10 GWh.

In [ ]:
print("=== Filter 1: Minimum area ===\n")

# Connected-component labeling
suit_binary = valid.astype(int)                                            # Binary: 1 = suitable
labeled, n_clusters = label(suit_binary)                                   # Label connected regions
print(f"  Connected areas before filter: {n_clusters}")

# Vectorized area calculation (bincount is O(n) — much faster than Python loop)
counts = np.bincount(labeled.ravel())                                      # Pixel count per cluster
areas_ha = counts * PX_AREA_HA                                            # Area in ha per cluster

# Filter: keep only clusters ≥ MIN_AREA_HA
keep_ids = np.where(areas_ha[1:] >= MIN_AREA_HA)[0] + 1                   # IDs passing filter (+1: skip background)
mask_area = np.isin(labeled, keep_ids).astype(np.uint8)                    # Boolean mask: 1 = kept
kept = len(keep_ids)                                                       # Number of clusters kept

print(f"  After ≥{MIN_AREA_HA} ha: {kept} kept, {n_clusters - kept} removed")
print(f"  Remaining area: {mask_area.sum() * PX_AREA_KM2:.0f} km² ({mask_area.sum() * PX_AREA_HA:.0f} ha)")

## 5. Filter 2: Min. 10 GWh Annual Production

Per contiguous area: Area × mean irradiation × η ≥ 10 GWh

Ref: Art. 71a Abs. 2 EnG. Frischholz et al. (2024): ~1400 kWh/kWp.

In [ ]:
print("=== Filter 2: Minimum 10 GWh/a ===\n")

labeled2, n2 = label(mask_area)                                            # Re-label after area filter
mask_gwh = np.zeros_like(mask_area)                                        # Init output mask

gwh_stats = []                                                             # Collect per-cluster stats
for i in range(1, n2 + 1):                                                # Loop through clusters
    cluster_mask = labeled2 == i                                           # Boolean mask for cluster i
    px_count = cluster_mask.sum()                                          # Pixel count
    area_m2 = px_count * PX_AREA_M2                                       # Area in m²

    solar_vals = solar_abs[cluster_mask & (solar_abs != NODATA)]           # Valid solar values in cluster
    if solar_vals.size == 0:                                               # Skip if no valid data
        continue
    mean_irrad = solar_vals.mean()                                         # Mean irradiation [kWh/m²/a]

    annual_gwh = area_m2 * mean_irrad * ETA / 1e6                          # Annual production [GWh]

    gwh_stats.append({                                                     # Store stats
        "cluster": i, "area_ha": px_count * PX_AREA_HA,
        "mean_irrad_kwh": mean_irrad, "annual_gwh": annual_gwh,
        "passes": annual_gwh >= MIN_ANNUAL_GWH
    })

    if annual_gwh >= MIN_ANNUAL_GWH:                                       # Keep if ≥ threshold
        mask_gwh[cluster_mask] = 1                                         # Mark as passing

df_gwh = pd.DataFrame(gwh_stats)                                          # Summary table
passed = df_gwh["passes"].sum()                                            # Count passing clusters
print(f"  After ≥{MIN_ANNUAL_GWH} GWh/a: {passed}/{len(df_gwh)} areas passed")
print(f"  Remaining area: {mask_gwh.sum() * PX_AREA_KM2:.0f} km²")
if len(df_gwh):
    print(f"  Production range: {df_gwh['annual_gwh'].min():.1f}–{df_gwh['annual_gwh'].max():.1f} GWh/a")

## 6. Filter 3: Min. 500 kWh/kW Winter Production

Per contiguous area: mean winter irradiation × η ≥ 500 kWh/kWp

Ref: Art. 71a Abs. 2 EnG. Frischholz et al. (2024): ~50% winter share alpine.

In [ ]:
print("=== Filter 3: Minimum 500 kWh/kW winter ===\n")

labeled3, n3 = label(mask_gwh)                                             # Re-label after GWh filter
mask_winter = np.zeros_like(mask_gwh)                                      # Init output mask

winter_stats = []                                                          # Collect per-cluster stats
for i in range(1, n3 + 1):                                                # Loop through clusters
    cluster_mask = labeled3 == i                                           # Boolean mask for cluster i

    winter_vals = winter_abs[cluster_mask & (winter_abs != NODATA)]        # Valid winter values
    if winter_vals.size == 0:                                              # Skip if no valid data
        continue
    mean_winter = winter_vals.mean()                                       # Mean winter irradiation [kWh/m²]

    # Winter production per kWp
    # BFE 75° data are annual values at 75° tilt
    # Alpine winter proxy: ~60% falls in Oct–Mar
    winter_kwh_kwp = mean_winter * ETA * 0.6 * 1000                        # kWh/kWp (×1000: kWp → m²)

    winter_stats.append({                                                  # Store stats
        "cluster": i, "area_ha": cluster_mask.sum() * PX_AREA_HA,
        "mean_winter_kwh": mean_winter,
        "winter_kwh_kwp": winter_kwh_kwp,
        "passes": winter_kwh_kwp >= MIN_WINTER_KWH_KW
    })

    if winter_kwh_kwp >= MIN_WINTER_KWH_KW:                               # Keep if ≥ threshold
        mask_winter[cluster_mask] = 1                                      # Mark as passing

df_winter = pd.DataFrame(winter_stats)                                     # Summary table
passed_w = df_winter["passes"].sum()                                       # Count passing clusters
print(f"  After ≥{MIN_WINTER_KWH_KW} kWh/kW: {passed_w}/{len(df_winter)} areas passed")
print(f"  Remaining area: {mask_winter.sum() * PX_AREA_KM2:.0f} km²")

## 7. Final Suitability Map & Top Sites

In [ ]:
print("=== Final suitability map ===\n")

# Apply all filters: WLC values only where all filters passed
suit_final = np.where(mask_winter == 1, suit, 0).astype(np.float32)        # Filtered suitability

# Save filtered raster
final_path = OUT / "suitability_postprocessed.tif"                         # Output path
with rasterio.open(final_path, "w", **profile) as dst:                     # Write GeoTIFF
    dst.write(suit_final, 1)                                               # Write band 1

print(f"  ✓ {final_path.name}")
print(f"  Suitable pixels: {(suit_final > 0).sum():,} ({(suit_final > 0).sum() * PX_AREA_KM2:.0f} km²)")

In [ ]:
print("\n=== Extracting top sites ===\n")

# Label final connected areas
labeled_final, n_final = label((suit_final > 0).astype(int))               # Connected components
print(f"  Final areas: {n_final}")

sites = []                                                                 # Collect site attributes
for i in range(1, n_final + 1):                                           # Loop through clusters
    cmask = labeled_final == i                                             # Boolean mask for cluster i
    px = cmask.sum()                                                       # Pixel count
    area_ha = px * PX_AREA_HA                                              # Area in hectares
    mean_suit = suit_final[cmask].mean()                                   # Mean suitability score
    max_suit  = suit_final[cmask].max()                                    # Max suitability score

    # Centroid → LV95 coordinates
    rows, cols = np.where(cmask)                                           # Pixel indices
    cy, cx = rows.mean(), cols.mean()                                      # Mean row/col
    ey = transform.f + cy * transform.e                                    # Northing [m]
    ex = transform.c + cx * transform.a                                    # Easting [m]

    # Annual production estimate
    s_vals = solar_abs[cmask & (solar_abs != NODATA)]                      # Solar values in cluster
    annual_gwh = (px * PX_AREA_M2 * s_vals.mean() * ETA / 1e6) if s_vals.size else 0  # GWh/a

    # Winter production estimate
    w_vals = winter_abs[cmask & (winter_abs != NODATA)]                    # Winter values in cluster
    winter_gwh = (px * PX_AREA_M2 * w_vals.mean() * ETA / 1e6) if w_vals.size else 0  # GWh winter

    sites.append({                                                         # Append to list
        "id": i, "area_ha": round(area_ha, 1), "area_km2": round(area_ha / 100, 2),
        "mean_suit": round(mean_suit, 3), "max_suit": round(max_suit, 3),
        "annual_gwh": round(annual_gwh, 1), "winter_gwh": round(winter_gwh, 1),
        "gwh_per_ha": round(annual_gwh / max(area_ha, 0.001), 3),
        "E_lv95": round(ex), "N_lv95": round(ey),
    })

df_sites = pd.DataFrame(sites).sort_values("mean_suit", ascending=False)   # Sort by suitability
df_sites = df_sites.reset_index(drop=True)                                 # Reset index
df_sites.index += 1                                                        # 1-based ranking
df_sites.index.name = "Rang"                                               # Index label
print(f"\nTop sites:")
df_sites.head(20)

In [ ]:
print("\n=== Exporting site polygons ===\n")

polygons = []                                                              # Collect polygon features
for geom, val in rio_shapes(labeled_final.astype(np.int32),                # Vectorize raster
                            mask=(labeled_final > 0), transform=transform):
    cid = int(val)                                                         # Cluster ID
    site_row = df_sites[df_sites["id"] == cid]                             # Match to stats table
    if site_row.empty:                                                     # Skip unmatched
        continue
    sr = site_row.iloc[0]                                                  # First matching row
    polygons.append({                                                      # Build feature dict
        "geometry": shp_shape(geom),                                       # Polygon geometry
        "id": cid, "area_ha": sr["area_ha"], "mean_suit": sr["mean_suit"],
        "suit_class": mean_suit_to_class(sr["mean_suit"]),                 # Suitability class (1–5)
        "annual_gwh": sr["annual_gwh"], "winter_gwh": sr["winter_gwh"],
        "gwh_per_ha": sr["gwh_per_ha"], "rang": site_row.index[0],
    })

gdf_sites = gpd.GeoDataFrame(polygons, crs=CRS)                           # Create GeoDataFrame
sites_path = OUT / "top_sites.gpkg"                                        # Output path
gdf_sites.to_file(sites_path, driver="GPKG")                              # Write GeoPackage
print(f"  ✓ {sites_path} ({len(gdf_sites)} features)")

## 8. Statistics Export

In [ ]:
print("=== Summary ===\n")

# CSV export
csv_path = OUT / "tables/postprocessing_stats.csv"                         # Output path
df_sites.to_csv(csv_path)                                                  # Write CSV
print(f"  ✓ {csv_path}")

# Summary table
print(f"\n{'='*55}")
print(f"  POST-PROCESSING SUMMARY")
print(f"{'='*55}")
print(f"  Before filters:       {valid.sum() * PX_AREA_KM2:>7.0f} km²")
print(f"  After ≥{MIN_AREA_HA} ha:           {mask_area.sum() * PX_AREA_KM2:>7.0f} km²")
print(f"  After ≥{MIN_ANNUAL_GWH} GWh/a:         {mask_gwh.sum() * PX_AREA_KM2:>7.0f} km²")
print(f"  After ≥{MIN_WINTER_KWH_KW} kWh/kW:       {mask_winter.sum() * PX_AREA_KM2:>7.0f} km²")
print(f"  Final sites:          {n_final:>7d}")
print(f"  Final area:           {(suit_final > 0).sum() * PX_AREA_KM2:>7.0f} km²")
if len(df_sites):
    print(f"  Est. production:      {df_sites['annual_gwh'].sum():>7.0f} GWh/a")
print(f"{'='*55}")

## 9. Maps

In [ ]:
BG_COLOR = "#6161AD"                                                       # Background color

# Canton border
gr_border = gpd.read_file(                                                 # Load GR boundary
    RAW / "swissboundaries/graubuenden_kantonsgrenze.shp").to_crs(CRS)

# Map extent in world coordinates
extent = [transform.c, transform.c + width * transform.a,                 # [xmin, xmax, ymin, ymax]
          transform.f + height * transform.e, transform.f]

# Classify all three stages
cls_before = suit_to_class(suit, valid)                                    # Before any filter
cls_area   = suit_to_class(suit, mask_area.astype(bool))                   # After area filter
cls_final  = suit_to_class(suit_final, (suit_final > 0))                   # After all filters

fig, axes = plt.subplots(1, 3, figsize=(20, 7), constrained_layout=True,  # 3-panel figure
                         facecolor=BG_COLOR)
fig.suptitle("Post-Processing — Art. 71a EnG Filter",                     # Title
             fontsize=13, fontweight="bold", color="white")

titles = [                                                                 # Panel titles with stats
    f"Before filters ({valid.sum() * PX_AREA_KM2:.0f} km²)",
    f"After ≥{MIN_AREA_HA} ha ({mask_area.sum() * PX_AREA_KM2:.0f} km²)",
    f"Final ({(suit_final > 0).sum() * PX_AREA_KM2:.0f} km², {n_final} sites)",
]
arrays = [cls_before, cls_area, cls_final]                                 # Data arrays

for ax, arr, title in zip(axes, arrays, titles):                           # Plot each panel
    ax.set_facecolor(BG_COLOR)                                             # Background color
    ax.imshow(arr, cmap=cmap_cls, norm=norm_cls, aspect="equal",           # Show classified raster
              extent=extent, interpolation="nearest")
    gr_border.boundary.plot(ax=ax, color="white", linewidth=0.8)           # Canton border
    ax.set_title(title, fontsize=10, color="white")                        # Panel title
    ax.set_axis_off()                                                      # Hide axes

# Shared legend
sm = plt.cm.ScalarMappable(cmap=cmap_cls, norm=norm_cls)                   # Colorbar source
sm.set_array([])                                                           # Required for ScalarMappable
cbar = fig.colorbar(sm, ax=axes, shrink=0.55, pad=0.04, aspect=30)        # Add colorbar
cbar.set_ticks([1, 2, 3, 4, 5])                                           # Tick positions
cbar.set_ticklabels(CLASS_LABELS)                                          # Tick labels
cbar.ax.yaxis.set_tick_params(color="white")                               # White tick marks
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")                   # White tick labels

fig.savefig(OUT / "figures/postprocessing_map.png",                        # Save figure
            dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
print(f"✓ Saved: {OUT / 'figures/postprocessing_map.png'}")

In [ ]:
print("\n=== Top sites map ===\n")

import math                                                                # For angle calculations

fig, ax = plt.subplots(figsize=(13, 10), constrained_layout=True,          # Single-panel figure
                       facecolor=BG_COLOR)
ax.set_facecolor(BG_COLOR)                                                 # Background color
gr_border.boundary.plot(ax=ax, color="white", linewidth=0.9)               # Canton border

top10 = gdf_sites.nlargest(10, "mean_suit")                                # Top 10 by suitability

# All sites (transparent), top-10 highlighted
gdf_sites.plot(ax=ax, column="suit_class", cmap=cmap_cls, norm=norm_cls,   # All sites (faded)
               alpha=0.6, edgecolor="none", linewidth=0)
top10.plot(ax=ax, column="suit_class", cmap=cmap_cls, norm=norm_cls,       # Top-10 (solid + border)
           alpha=1.0, edgecolor="black", linewidth=1.2)

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap_cls, norm=norm_cls)                   # Colorbar source
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.5, pad=0.04, fraction=0.03)       # Add colorbar
cbar.set_ticks([1, 2, 3, 4, 5])                                           # Tick positions
cbar.set_ticklabels(CLASS_LABELS)                                          # Tick labels
cbar.set_label("Suitability class", color="white")                         # Colorbar label
cbar.ax.yaxis.set_tick_params(color="white")                               # White styling
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")

# Label placement (radial from map center, collision avoidance)
xlims, ylims = ax.get_xlim(), ax.get_ylim()                               # Map extent
map_cx = (xlims[0] + xlims[1]) / 2                                        # Map center X
map_cy = (ylims[0] + ylims[1]) / 2                                        # Map center Y
LABEL_DIST = 18_000                                                        # Initial label distance [m]
MIN_SEP    = 14_000                                                        # Min. label separation [m]

placed = []                                                                # Placed label positions
for _, r in top10.sort_values("rang").iterrows():                          # Loop top-10 by rank
    c = r.geometry.centroid                                                # Polygon centroid
    cx, cy = c.x, c.y                                                     # Centroid coordinates
    base_angle = math.atan2(cy - map_cy, cx - map_cx)                      # Angle from map center

    # Find non-overlapping position (rotate in 15° steps)
    dist = LABEL_DIST                                                      # Start distance
    best_ox, best_oy = None, None                                          # Best position
    for attempt in range(2):                                               # Two distance attempts
        for step in range(24):                                             # 24 × 15° = full circle
            angle = base_angle + step * math.pi / 12                       # Candidate angle
            ox = cx + math.cos(angle) * dist                               # Candidate X
            oy = cy + math.sin(angle) * dist                               # Candidate Y
            if all(math.hypot(ox-px, oy-py) >= MIN_SEP for px, py in placed):  # Check separation
                best_ox, best_oy = ox, oy                                  # Accept position
                break
        if best_ox is not None:                                            # Found good position
            break
        dist += 8_000                                                      # Increase radius

    if best_ox is None:                                                    # Fallback: default angle
        best_ox = cx + math.cos(base_angle) * dist
        best_oy = cy + math.sin(base_angle) * dist

    placed.append((best_ox, best_oy))                                      # Register position
    ax.annotate(                                                           # Draw label
        f"#{r['rang']} ({r['area_ha']:.0f} ha)",                           # Label text
        xy=(cx, cy), xytext=(best_ox, best_oy),                            # From → To
        fontsize=8, color="white", fontweight="bold",
        arrowprops=dict(arrowstyle="-", color="white", lw=0.6),            # Arrow style
        bbox=dict(boxstyle="round,pad=0.2", fc=BG_COLOR, ec="white", alpha=0.85, lw=0.5))

ax.set_title(f"Top Sites — {n_final} areas after post-processing",        # Title
             fontsize=12, fontweight="bold", color="white")
ax.set_axis_off()                                                          # Hide axes
fig.savefig(OUT / "figures/top_sites_map.png",                             # Save figure
            dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
print(f"✓ Saved: {OUT / 'figures/top_sites_map.png'}")

## Next Step

The filtered suitability map and top site polygons are ready for:

→ **05_detail_swissalti3d.ipynb** — Micro-siting with 2m DEM for top-10 sites
→ **06_validation.ipynb** — Hit rate, Mann-Whitney U, Kruskal-Wallis + OAT sensitivity